Recipe for plotting the `results.json` file generated by:
```
 ./run-scan.py --minThreads 1 --maxThreads 5 --numStreams 1,3,5,7,10,12,15 --eventsPerStream 200 --cudaDevices 0 ./alpaka --cuda
 ```


In [6]:
%%python
import ROOT
import json

# Convert the json file to a CSV, just because I ahve the plots from wp1.7-scheduler-tests handy.

# Open and read the JSON file
with open('result.json', 'r') as file:
    data = json.load(file)

# conver to csv to reuse the plots from wp1.7-scheduler-tests
print(data["results"])
min = None
max = None
for r in data["results"]:
    if min is None or r['throughput'] < min:
        min = r['throughput']
    if max is None or r['throughput'] > max:
        max = r['throughput']
    print(f"Min Throughput: {min}, Max Throughput: {max}")
for r in data["results"]:
    # print rows: 'threads': 1, 'streams': 1, 'events': 200, 'throughput': 429.871, 'cpueff': 143.6
    print(f"Threads: {r['threads']}, Streams: {r['streams']}, Events: {r['events']}, Throughput: {r['throughput']}, CPU Efficiency: {r['cpueff']}")

with open('results.csv', 'w') as f:
    f.write("Slots,Threads,Rate\n")
    for r in data["results"]:
        f.write(f"{r['streams']},{r['threads']},{r['throughput']}\n")


[{'hostname': 'cano-desk', 'threads': 1, 'streams': 1, 'events': 200, 'throughput': 429.871, 'cpueff': 143.6, 'cudaDevices': {'0': {'name': 'NVIDIA GeForce RTX 5070', 'driver_version': '576.40'}}}, {'hostname': 'cano-desk', 'threads': 1, 'streams': 3, 'events': 600, 'throughput': 1044.88, 'cpueff': 161.7, 'cudaDevices': {'0': {'name': 'NVIDIA GeForce RTX 5070', 'driver_version': '576.40'}}}, {'hostname': 'cano-desk', 'threads': 1, 'streams': 5, 'events': 1000, 'throughput': 1184.88, 'cpueff': 171.6, 'cudaDevices': {'0': {'name': 'NVIDIA GeForce RTX 5070', 'driver_version': '576.40'}}}, {'hostname': 'cano-desk', 'threads': 1, 'streams': 7, 'events': 1400, 'throughput': 1164.98, 'cpueff': 170.1, 'cudaDevices': {'0': {'name': 'NVIDIA GeForce RTX 5070', 'driver_version': '576.40'}}}, {'hostname': 'cano-desk', 'threads': 1, 'streams': 10, 'events': 2000, 'throughput': 1197.56, 'cpueff': 171.9, 'cudaDevices': {'0': {'name': 'NVIDIA GeForce RTX 5070', 'driver_version': '576.40'}}}, {'hostname

In [5]:
#include <ROOT/RDataFrame.hxx>
#include <numeric>

// Adjust the path to your CSV file
//std::string buildDir = "../../wp1.7-scheduler-tests/";
std::string csvfile = "results.csv";
system((std::string("ls -l ")+csvfile).c_str());


// Load CSV with columns: Slots,Threads,Rate
auto df = ROOT::RDF::FromCSV(csvfile);

std::cout << df.Count().GetValue() << " rows loaded from " << csvfile << std::endl;
std::cout << "Columns: ";
for (auto & c: df.GetColumnNames())
  std::cout << c << " ";
std::cout << std::endl;

auto cRate = new TCanvas("cRate", "Rate vs Threads and Slots", 1600, 800);
cRate->Divide(2, 1);
auto gflat = new TGraph2D();
auto g3d = new TGraph2D();
gflat->SetTitle("Rate vs Threads and Slots (pre warmup);Slots;Threads;Rate (evts/s)");
g3d->SetTitle("Rate vs Threads and Slots (pre warmup);Slots;Threads;Rate (evts/s)");

int i=0;
df.Foreach([&](double b, Long64_t s, Long64_t t){gflat->SetPoint(i, s, t, b);g3d->SetPoint(i++, s, t, b);}, {"Rate", "Slots", "Threads"});
gflat->SetMinimum(minRate);
gflat->SetMaximum(maxRate);
g3d->SetMinimum(minRate);
g3d->SetMaximum(maxRate);
cRate->cd(1);
gflat->Draw("COLZ RY");
cRate->cd(2);
g3d->Draw("SURF1 RY");
gStyle->SetPalette(1);
cRate->Draw();

-rw-r--r-- 1 cano cano 452 Jun 13 16:45 results.csv
35 rows loaded from results.csv
Columns: Rate Slots Threads 


Warning in <TCanvas::Constructor>: Deleting canvas with same name: cRate
